In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pyarrow
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

#Desactivar notacion científica
pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
Path.cwd()

WindowsPath('g:/Mi unidad/DS4B_Mastery_Edition/proyectos/ecommerce-growth-analytics/notebooks')

In [18]:
df = pd.read_parquet("../data/intermediate/df_clean.parquet")

#### Creacion variables temporales

In [20]:
# Eliminar zona horaria (+00:00)
df["date_time"] = df["date"].dt.tz_localize(None)

# Crear variables temporales
df["date"] = df["date_time"].dt.normalize()
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month_name().astype("category")
df["day"] = df["date_time"].dt.day
df["day_of_week"] = df["date_time"].dt.day_name().astype("category")
df["hour"] = df["date_time"].dt.hour

In [21]:
df.set_index('date_time', inplace=True)

In [22]:
# Creacion variable holidays con dias festivos rusos
import holidays

years = sorted([2019,2020])
f = holidays.RU(years=years)
holiday_dates = pd.Index(pd.to_datetime(list(f.keys())))
df["holiday"] = np.where(df["date"].isin(holiday_dates), 1, 0)

In [23]:
df.holiday.value_counts()

holiday
0    1940833
1     133193
Name: count, dtype: int64

In [24]:
#variables exógenas temporales de valor para Rusia
df["is_unity_day"] = (df["date"] == "2019-11-04").astype("int8")
df["is_singles_day"] = (df["date"] == "2019-11-11").astype("int8")
df["is_black_friday"] = (df["date"].between("2019-11-22", "2019-11-29")).astype("int8")
df["is_cyber_monday"] = (df["date"].between("2020-01-27", "2020-01-29")).astype("int8")
df["is_new_year_period"] = (df["date"].between("2019-12-31", "2020-01-08")).astype("int8")
df["is_orthodox_christmas"] = (df["date"] == "2020-01-07").astype("int8")
df["is_valentines_day"] = (df["date"] == "2020-02-14").astype("int8")
df["is_defender_day"] = (df["date"] == "2020-02-23").astype("int8")

Reordenar variables dataset

In [25]:
vars = df.columns.to_list()

In [26]:
order = ['user_id',
         'user_session',
         'event',
         'product_id',
         'category',
         'price']

vars_new = order + [var for var in vars if var not in order]

In [27]:
print(vars_new)

['user_id', 'user_session', 'event', 'product_id', 'category', 'price', 'date', 'year', 'month', 'day', 'day_of_week', 'hour', 'holiday', 'is_unity_day', 'is_singles_day', 'is_black_friday', 'is_cyber_monday', 'is_new_year_period', 'is_orthodox_christmas', 'is_valentines_day', 'is_defender_day']


In [28]:
df = df[vars_new]

In [29]:
df.to_parquet("../data/intermediate/df_processed.parquet", index=True)